# Boston Housing — Entrenamiento y Selección de Modelos

In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

SQLITE_PATH = project_root / os.environ.get("SQLITE_PATH", ".storage/data.db")
TABLE_NAME = "housing_data"
MLFLOW_TRACKING_URI = "http://localhost:5001"
MLFLOW_S3_ENDPOINT_URL = "http://minio:9000"

EXPERIMENT_NAME = "boston-housing-experiments"

os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("MINIO_ROOT_USER", "")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("MINIO_ROOT_PASSWORD", "")
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ.get("MINIO_ENDPOINT", "")


print(f"MLflow URI: {MLFLOW_TRACKING_URI}")
print(f"MinIO endpoint: {MLFLOW_S3_ENDPOINT_URL}")

MLflow URI: http://localhost:5001
MinIO endpoint: http://minio:9000


In [11]:
print(os.environ["AWS_ACCESS_KEY_ID"])
print(os.environ["AWS_SECRET_ACCESS_KEY"])
print(os.environ["MLFLOW_S3_ENDPOINT_URL"])

admin12345
admin12345



In [2]:
# Dependencias necesarias
import sqlite3
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.data.repository import load_data
from src.data.schema import FEATURE_COLUMNS, TARGET_COLUMN

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [3]:
# Carga inicial de datos
df = load_data(SQLITE_PATH, "housing_data")
print("Tamaño:", df.shape)
df.head()

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Tamaño: (506, 15)
Train: (455, 13), Test: (51, 13)


In [4]:
# Evaluar modelo
def evaluate(pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "mae": float(mean_absolute_error(y_test, y_pred)),
        "r2": float(r2_score(y_test, y_pred)),
    }

# Construir preprocesador
def build_preprocessor():
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    return ColumnTransformer(
        transformers=[("num", numeric_pipeline, FEATURE_COLUMNS)],
        remainder="drop",
        verbose_feature_names_out=False,
    )


In [5]:
# Experimento en MlFlow
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experiment: {EXPERIMENT_NAME}")

Experiment: boston-housing-experiments


In [6]:
# Entrenamiento de con Ridge
ridge_params = {"alpha": 1.0, "random_state": 42}

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", build_preprocessor()),
    ("estimator", Ridge(**ridge_params)),
])

with mlflow.start_run(run_name="ridge") as run:
    mlflow.set_tag("model_name", "ridge")
    mlflow.log_params(ridge_params)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.2)

    ridge_pipeline.fit(X_train, y_train)
    ridge_metrics = evaluate(ridge_pipeline, X_test, y_test)
    mlflow.log_metrics(ridge_metrics)

    mlflow.sklearn.log_model(
        sk_model=ridge_pipeline,
        artifact_path="model",
        input_example=X_train.iloc[:3],
    )

    ridge_run_id = run.info.run_id

print(f"Ridge run_id: {ridge_run_id}")
print(f"Ridge metrics: {ridge_metrics}")

2026/05/12 18:47:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 18:47:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/Users/dalzoj/Documents/IA/boston-housing-mlops/.venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that inclu

🏃 View run ridge at: http://localhost:5001/#/experiments/2/runs/50979e103c19459a8fc22c094dfdd30f
🧪 View experiment at: http://localhost:5001/#/experiments/2


S3UploadFailedError: Failed to upload /var/folders/wv/4s2g1dhx5yx7ybggv67jlt140000gn/T/tmptgwx9fkz/model/python_env.yaml to mlflow/2/models/m-d8a398a702ee48ec9e56ee72554603f3/artifacts/python_env.yaml: An error occurred (InvalidAccessKeyId) when calling the PutObject operation: The AWS Access Key Id you provided does not exist in our records.